# 0 - Téléchargement des données

Ce notebook illustre les différentes fonctionnalités des clients du répertoire pour télécharger des données via l'API SDMX. Chaque client permet de télécharger des données depuis une source distincte (Eurostat etc ...)

## Table des matières

0. [Importation des modules](#section-0)
1. [Téléchargement des données d'Eurostat](#section-1)
    - 1.1 [Catalogue des dataflows disponibles (list_all_dataflows)](#section-1.1)
    - 1.2 [Description de la structure d'un dataflow (get_dataflow_structure)](#section-1.2)
    - 1.3 [Requêtes de structure SDMX (get_structure)](#section-1.3)
    - 1.4 [Requête basique avec noms de dimensions](#section-1.4)
    - 1.5 [Filtres temporels](#section-1.5)
    - 1.6 [Formats de réponse](#section-1.6)
    - 1.7 [Séparation des requêtes avec split dimensions](#section-1.7)
    - 1.8 [Utilisation de EurostatQueryRequest](#section-1.8)
    - 1.9 [Datasets Comext (DS-*)](#section-1.9)
    - 1.10 [Utilisation en tant que context manager](#section-1.10)
    - 1.11 [Gestion des erreurs](#section-1.11)
    - 1.12 [Memento et conseils de performance](#section-1.12)


## 0 - Importation des modules <a id="section-0"></a>

Importation des modules nécessaires et configuration de l'environnement.

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import io
import sys
import xml.etree.ElementTree as ET
import re
import pandas as pd
from pathlib import Path

# Ajout du répertoire parent au path
sys.path.append('..')

# Modules du package
from macroforecast.datasets.sources import (
    EurostatClient,
    EurostatQueryRequestV30,
    EurostatResponseFormat
)
from macroforecast.datasets.core.sdmx import StructureResourceType

## 1 - Téléchargement des données d'Eurostat <a id="section-1"></a>

In [ ]:
# Initialisation du client
client = EurostatClient()

### 1.1 - Catalogue des dataflows disponibles <a id="section-1.1"></a>

La méthode `list_all_dataflows()` récupère le catalogue complet des dataflows disponibles sur l'endpoint configuré. Elle retourne un `DataFrame` avec les colonnes `id`, `name`, `version` et `agency`.

En interne, elle appelle `get_structure()` avec `resource_id="*"` et `detail="allstubs"`, ce qui déclenche le *Dataset listing* spécial de l'API Eurostat SDMX :

| Version API | Endpoint ciblé |
|---|---|
| SDMX 3.0 | `/sdmx/3.0/structure/dataflow/{agency}/*` |
| SDMX 2.1 | `/sdmx/2.1/dataflow/{agency}/all/latest` |

Le paramètre `agency` permet de filtrer par agence : `"*"` (défaut) retourne tous les dataflows, `"ESTAT"` restreint aux datasets officiels Eurostat.


In [ ]:
# Récupération du catalogue complet des dataflows (toutes agences)
catalogue = client.list_all_dataflows()

# Affichage
print(f"{len(catalogue)} dataflow(s) disponible(s) au total")

catalogue.head(10)


In [ ]:
# Restriction aux datasets officiels Eurostat uniquement (agency='ESTAT')
catalogue_estat = client.list_all_dataflows(agency="ESTAT")

# Affichage
print(f"{len(catalogue_estat)} dataflow(s) Eurostat officiel(s)")

catalogue_estat.head(10)

### 1.2 - Description de la structure d'un dataflow <a id="section-1.2"></a>

La méthode `get_dataflow_structure()` permet de récupérer et de parser la DSD (*Data Structure Definition*) d'un dataflow. Elle retourne un objet `DataflowStructure` contenant la liste des dimensions avec leur position et leur description. Contrairement au client OCDE, l'agence est toujours `ESTAT` et n'est pas un paramètre.

In [ ]:
# Récupération et parsing de la DSD du dataflow namq_10_gdp (comptes nationaux trimestriels)
structure = client.get_dataflow_structure(
    dataflow="namq_10_gdp",
    version="+",
)

# Affichage
print(f"Agency: {structure.agency}")
print(f"Dataflow: {structure.dataflow}")
print(f"Nombre de dimensions: {structure.num_dimensions}")
print("\nDimensions:")
for dim in structure.dimensions:
    print(f"  Position {dim.position}: {dim.name} - {dim.description}")

### 1.3 - Requêtes de structure SDMX <a id="section-1.3"></a>

La méthode `get_structure()` est le point d'entrée unique pour interroger l'API SDMX et récupérer des artefacts de structure. Elle retourne le XML brut de la réponse SDMX-ML 3.0.

Le paramètre `resource_type` de type `StructureResourceType` détermine l'endpoint interrogé :

| Type | Endpoint SDMX | Description |
|---|---|---|
| `DATAFLOW` | `/structure/dataflow/...` | Définition du dataflow (nom, référence à la DSD) |
| `DATASTRUCTURE` | `/structure/datastructure/...` | DSD : dimensions, attributs, mesures et références aux codelists |
| `DATACONSTRAINT` | `/structure/dataconstraint/...` | Contraintes de contenu : combinaisons de valeurs autorisées |
| `CONCEPTSCHEME` | `/structure/conceptscheme/...` | Schémas de concepts : définitions des concepts de la DSD |
| `CODELIST` | `/structure/codelist/...` | Listes de codes : vocabulaire contrôlé des valeurs d'une dimension |

Le paramètre `references` permet de récupérer les artefacts liés en un seul appel :
- `"none"` (défaut) : artefact ciblé uniquement
- `"children"` : artefact + artefacts directement référencés
- `"descendants"` : artefact + tous les artefacts référencés récursivement

Les exemples suivants utilisent les namespaces SDMX 3.0 pour parser le XML retourné.


In [ ]:
# Namespaces SDMX 3.0 pour le parsing XML (commun à tous les exemples de cette section)
NS_30 = {
    "mes": "http://www.sdmx.org/resources/sdmxml/schemas/v3_0/message",
    "str": "http://www.sdmx.org/resources/sdmxml/schemas/v3_0/structure",
    "com": "http://www.sdmx.org/resources/sdmxml/schemas/v3_0/common",
}

In [ ]:
# Endpoint DATAFLOW : définition et métadonnées du dataflow namq_10_gdp
xml_text = client.get_structure(
    resource_type=StructureResourceType.DATAFLOW,
    resource_id="namq_10_gdp",
)

# Extraction des métadonnées du dataflow
root = ET.fromstring(xml_text)
df_elem = root.find(".//str:Dataflow", NS_30)

if df_elem is not None:
    print(f"ID      : {df_elem.get('id')}")
    print(f"Agency  : {df_elem.get('agencyID')}")
    print(f"Version : {df_elem.get('version')}")
    name_elem = df_elem.find("com:Name", NS_30)
    if name_elem is not None:
        print(f"Nom     : {name_elem.text}")
    # Référence à la DSD associée (utile pour les appels DATASTRUCTURE)
    ref = df_elem.find(".//str:Structure/com:Ref", NS_30)
    if ref is not None:
        print(f"\nDSD associée : {ref.get('id')} (agencyID={ref.get('agencyID')})")

In [ ]:
# Endpoint DATASTRUCTURE : DSD du dataflow (identifiant en majuscules)
# Note : pour namq_10_gdp, la DSD porte l'identifiant NAMQ_10_GDP
xml_text = client.get_structure(
    resource_type=StructureResourceType.DATASTRUCTURE,
    resource_id="NAMQ_10_GDP",
)

# Extraction de la liste des dimensions et de leurs références aux codelists
root = ET.fromstring(xml_text)
dsd_elem = root.find(".//str:DataStructure", NS_30)

if dsd_elem is not None:
    print(f"DSD : {dsd_elem.get('id')} v{dsd_elem.get('version')}\n")
    dim_list = dsd_elem.find(".//str:DimensionList", NS_30)
    if dim_list is not None:
        dims = dim_list.findall("str:Dimension", NS_30)
        print(f"{len(dims)} dimensions :")
        for dim in sorted(dims, key=lambda d: int(d.get("position", 0))):
            # Extraction de la liste de codes de la dimension
            cl_elem = dim.find(".//str:Enumeration", NS_30)
            cl_id = "—"
            if cl_elem is not None and cl_elem.text:
                # Format URN : ...Codelist=ESTAT:FREQ(3.9)
                match = re.search(r"=\w+:(\w+)\(", cl_elem.text)
                if match:
                    cl_id = match.group(1)
            print(f"  [{dim.get('position')}] {dim.get('id'):20s} → codelist: {cl_id}")

In [ ]:
# Endpoint DATACONSTRAINT : valeurs autorisées par dimension pour namq_10_gdp
xml_text = client.get_structure(
    resource_type=StructureResourceType.DATACONSTRAINT,
    resource_id="namq_10_gdp",
)

# Extraction des cubes de contrainte
root = ET.fromstring(xml_text)
constraint_elem = root.find(".//str:DataConstraint", NS_30)

if constraint_elem is not None:
    print(f"Contrainte : {constraint_elem.get('id')}\n")
    cube = constraint_elem.find(".//str:CubeRegion", NS_30)
    if cube is not None:
        key_values = cube.findall(".//str:KeyValue", NS_30)
        print("Valeurs autorisées par dimension :")
        for kv in key_values:
            dim_id = kv.get("id")
            values = [v.text for v in kv.findall("str:Value", NS_30)]
            display = str(values[:5])
            suffix = f" … ({len(values)} valeurs au total)" if len(values) > 5 else ""
            print(f"  {dim_id:20s}: {display}{suffix}")

In [ ]:
# Endpoint CODELIST : vocabulaire contrôlé d'une dimension
# Les identifiants des codelists suivent le motif <DIM> : GEO, NA_ITEM,
# UNIT, FREQ, etc. (récupérables depuis la DSD via DATASTRUCTURE)
xml_text = client.get_structure(
    resource_type=StructureResourceType.CODELIST,
    resource_id="GEO",
)

# Extraction des codes et de leurs libellés
root = ET.fromstring(xml_text)
cl_elem = root.find(".//str:Codelist", NS_30)

if cl_elem is not None:
    name_elem = cl_elem.find("com:Name", NS_30)
    print(f"Codelist : {cl_elem.get('id')}")
    if name_elem is not None:
        print(f"Nom      : {name_elem.text}")
    codes = cl_elem.findall("str:Code", NS_30)
    print(f"\n{len(codes)} code(s) — affichage des 10 premiers :")
    for code_elem in codes[:10]:
        desc = code_elem.find("com:Name", NS_30)
        print(f"  {code_elem.get('id'):15s}: {desc.text if desc is not None else ''}")

In [ ]:
# Metadata harvesting : tous les codelists Eurostat en un seul appel
# resource_id='*' déclenche l'endpoint spécial de metadata harvesting
# compress='true' recommandé : la réponse peut être très volumineuse
xml_all_codelists = client.get_structure(
    resource_type=StructureResourceType.CODELIST,
    resource_id="*",
    agency="ESTAT",
    compress="true",
)

# Listing des codelists disponibles
root_all = ET.parse(io.StringIO(xml_all_codelists)).getroot()
codelists = root_all.findall(".//str:Codelist", NS_30)
print(f"{len(codelists)} codelist(s) disponible(s) :\n")
for cl in codelists[:10]:
    name_elem = cl.find("com:Name", NS_30)
    name_text = name_elem.text if name_elem is not None else ""
    print(f"  {cl.get('id'):40s} {name_text}")


In [ ]:
# Endpoint CONCEPTSCHEME : listing de tous les schémas de concepts Eurostat
# resource_id='all' retourne l'ensemble des schémas disponibles
xml_text = client.get_structure(
    resource_type=StructureResourceType.CONCEPTSCHEME,
    resource_id="*",
    version=None,
    detail="allstubs", # "referencestubs" permet d'avoir de plus petites requêtes
    compress="true",
    timeout=300,
)

# Listing des schémas de concepts
root = ET.parse(io.StringIO(xml_text)).getroot()
schemes = root.findall(".//str:ConceptScheme", NS_30)
print(f"{len(schemes)} schéma(s) de concepts disponible(s) :\n")
for scheme in schemes[:10]:
    name_elem = scheme.find("com:Name", NS_30)
    name_text = name_elem.text if name_elem is not None else ""
    print(f"  {scheme.get('id'):40s} {name_text}")

In [ ]:
# Récupération d'un schéma de concepts spécifique (avec les concepts détaillés)
# L'identifiant est récupéré depuis le listing précédent
xml_text = client.get_structure(
    resource_type=StructureResourceType.CONCEPTSCHEME,
    resource_id="NAMQ_10_GDP",  # Remplacer par l'ID obtenu ci-dessus
    detail="full",
)

# Extraction des concepts du schéma
root = ET.fromstring(xml_text)
cs_elem = root.find(".//str:ConceptScheme", NS_30)

if cs_elem is not None:
    name_elem = cs_elem.find("com:Name", NS_30)
    print(f"ConceptScheme : {cs_elem.get('id')}")
    if name_elem is not None:
        print(f"Nom           : {name_elem.text}")
    concepts = cs_elem.findall("str:Concept", NS_30)
    print(f"\n{len(concepts)} concept(s) — affichage des 10 premiers :")
    for concept in concepts[:10]:
        desc = concept.find("com:Name", NS_30)
        print(f"  {concept.get('id'):20s}: {desc.text if desc is not None else ''}")

In [ ]:
# Paramètre references='descendants' : récupération d'un dataflow
# et de tous ses artefacts liés en un seul appel API
xml_text = client.get_structure(
    resource_type=StructureResourceType.DATAFLOW,
    resource_id="namq_10_gdp",
    references="descendants",
    detail="referencestubs",  # Stubs pour les artefacts référencés
)

# Comptage des artefacts présents dans la réponse
root = ET.fromstring(xml_text)
artefacts = {
    "Dataflow"      : root.findall(".//str:Dataflow", NS_30),
    "DataStructure" : root.findall(".//str:DataStructure", NS_30),
    "Codelist"      : root.findall(".//str:Codelist", NS_30),
    "ConceptScheme" : root.findall(".//str:ConceptScheme", NS_30),
}
print("Artefacts retournés avec references='descendants' :")
for artefact_type, elements in artefacts.items():
    print(f"  {artefact_type:20s} : {len(elements)}")

### 1.4 - Requête basique avec noms de dimensions <a id="section-1.4"></a>

La méthode `get_data()` permet de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur nom. Le paramètre `dimensions` accepte aussi bien une valeur unique (`str`) qu'une liste de valeurs (`List[str]`).

In [ ]:
# Requête sur le dataflow namq_10_gdp (PIB trimestriel)
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": ["FR", "DE"],  # France et Allemagne
        "FREQ": "Q",           # Fréquence trimestrielle
        "NA_ITEM": "B1GQ",     # PIB aux prix du marché
        "UNIT": "CP_MEUR",     # Millions d'euros courants
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Colonnes: {list(df.columns)}")
print(f"Pays: {sorted(df['geo'].unique())}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

### 1.5 - Filtres temporels <a id="section-1.5"></a>

La méthode `get_data()` propose plusieurs paramètres pour filtrer les observations dans le temps :
- `start_period` / `end_period` : borne inférieure et supérieure de la période (format SDMX, ex. `"2020-Q1"`, `"2024"`)
- `last_n_observations` : N dernières observations seulement
- `first_n_observations` : N premières observations seulement

In [ ]:
# Filtre par période de début et de fin
df_period = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": "FR",
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    #start_period="2022-Q1",
    #end_period="2024-Q4"
)

print("--- Filtre start_period / end_period ---")
print(f"Nombre de lignes: {len(df_period)}")
print(f"Période: {df_period['TIME_PERIOD'].min()} - {df_period['TIME_PERIOD'].max()}")

In [ ]:
# Récupération des N dernières observations uniquement
df_last = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": "FR",
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    last_n_observations=4  # Quatre derniers trimestres
)

print("--- last_n_observations=4 ---")
print(f"Nombre de lignes: {len(df_last)}")
df_last

### 1.6 - Formats de réponse <a id="section-1.6"></a>

L'API SDMX 3.0 d'Eurostat supporte plusieurs formats de réponse, encapsulés dans l'énumération `EurostatResponseFormat` :
- **CSV** (défaut) : SDMX-CSV 3.0, format tidy en colonnes
- **TSV** : format héritage Eurostat en colonnes larges avec flags
- **JSON** : JSON-stat 2.0
- **XML** : SDMX-ML 3.0 (données structurées)

In [ ]:
# Affichage des formats disponibles
print("Formats de réponse disponibles:")
for fmt in EurostatResponseFormat:
    print(f"  - {fmt.name}: '{fmt.value}'")

In [ ]:
# Paramètres communs à la requête de comparaison
common_kwargs = dict(
    dataflow="une_rt_m",  # Taux de chômage mensuel
    dimensions={
        "GEO": "FR",
        "FREQ": "M",
        "AGE": "TOTAL",
        "SEX": "T",
        "UNIT": "PC_ACT",
    },
    start_period="2024-01",
    end_period="2024-06",
)

# Requête au format CSV (défaut)
df_csv = client.get_data(**common_kwargs, format=EurostatResponseFormat.CSV)

# Requête au format JSON
df_json = client.get_data(**common_kwargs, format=EurostatResponseFormat.JSON)

# Comparaison des résultats
print("--- Format CSV ---")
print(f"Colonnes: {list(df_csv.columns)}")
print(df_csv[["geo", "TIME_PERIOD", "OBS_VALUE"]].head(3).to_string(index=False))

print("\n--- Format JSON ---")
print(f"Colonnes: {list(df_json.columns)}")
print(df_json.head(3).to_string(index=False))

### 1.7 - Séparation des requêtes avec split dimensions <a id="section-1.7"></a>

Le paramètre `split_dimensions` de la méthode `get_data()` permet d'effectuer une requête distincte pour chaque combinaison de valeurs des dimensions spécifiées. Cela permet de gérer les requêtes volumineuses ou de contourner les limitations de l'API.

In [ ]:
# Requête avec split_dimensions
# Au lieu d'une seule requête pour 6 pays, on génère 6 requêtes séparées
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": ["FR", "DE", "IT", "ES", "NL", "BE"],
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    start_period="2022",
    split_dimensions=["GEO"]  # Génère 6 requêtes séparées, une par pays
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['geo'].unique())}")
print("\nNombre d'observations par pays:")
print(df.groupby('geo').size())

### 1.8 - Utilisation de EurostatQueryRequest <a id="section-1.8"></a>

L'objet `EurostatQueryRequest` encapsule tous les paramètres d'une requête et peut être passé à `execute_query()`. Il facilite la manipulation programmatique des requêtes (stockage, modification, sérialisation).

In [ ]:
# Création d'une EurostatQueryRequest
query = EurostatQueryRequestV30(
    dataflow="namq_10_gdp",
    version="*",
    dimensions={
        "GEO": ["FR", "DE"],
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    start_period="2020"
)

# Affichage des métadonnées de la requête
print(f"Dataflow key: {query.get_dataflow_key()}")
print(f"Dimensions: {query.dimensions}")
print(f"Format: {query.format}")

# Exécution de la requête
df = client.execute_query(query)

# Affichage
print(f"\nNombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

In [ ]:
# Modification d'une QueryRequest existante (ex. ajout d'un pays)
query_extended = EurostatQueryRequestV30(
    **{
        **query.to_dict(),  # Reprise des paramètres existants
        "dimensions": {
            **query.dimensions,
            "GEO": ["FR", "DE", "IT"],  # Ajout de l'Italie
        },
        "last_n_observations": 8,       # Restriction aux 8 dernières observations
    }
)

print(f"Pays dans la requête étendue: {query_extended.dimensions['GEO']}")
print(f"Dernières observations: {query_extended.last_n_observations}")

# Exécution
df_extended = client.execute_query(query_extended)
print(f"\nNombre de lignes: {len(df_extended)}")

### 1.9 - Datasets Comext (DS-*) <a id="section-1.9"></a>

Les datasets dont l'identifiant commence par `DS-` proviennent de la base **Comext** (données de commerce extérieur). Le client détecte automatiquement ces datasets et bascule vers l'API Comext dédiée (`ec.europa.eu/eurostat/api/comext/...`). Aucune configuration supplémentaire n'est requise.

In [ ]:
# Vérification de la détection automatique Comext
comext_dataflow = "DS-045409"
standard_dataflow = "namq_10_gdp"

print("Détection automatique des datasets Comext:")
print(f"  '{comext_dataflow}' → Comext: {client._is_comext_dataset(comext_dataflow)}")
print(f"  '{standard_dataflow}' → Comext: {client._is_comext_dataset(standard_dataflow)}")

# Requête sur un dataset Comext (import/export par produit)
# La sélection du bon endpoint est transparente pour l'utilisateur
try:
    df_comext = client.get_data(
        dataflow="DS-045409",
        dimensions={
            "FLOW": "1",        # Importations
            "REPORTER": "FR",   # France
            "PRODUCT": "27",    # Produits énergétiques (section SH)
        },
        last_n_observations=4,
    )
    print(f"\nNombre de lignes récupérées (Comext): {len(df_comext)}")
    df_comext.head()
except Exception as e:
    print(f"\nErreur lors de la requête Comext: {type(e).__name__}: {str(e)[:120]}")

### 1.10 - Utilisation en tant que context manager <a id="section-1.10"></a>

`EurostatClient` implémente le protocole de context manager (`__enter__` / `__exit__`). Son utilisation avec `with` garantit la fermeture propre des connexions réseau, même en cas d'exception.

In [ ]:
# Utilisation recommandée : context manager
with EurostatClient() as eurostat:
    # Récupération de la structure
    structure = eurostat.get_dataflow_structure(dataflow="une_rt_m")
    print(f"Structure chargée pour '{structure.dataflow}' ({structure.num_dimensions} dimensions)")

    # Récupération des données
    df = eurostat.get_data(
        dataflow="une_rt_m",
        dimensions={
            "GEO": ["FR", "DE"],
            "FREQ": "M",
            "AGE": "TOTAL",
            "SEX": "T",
            "UNIT": "PC_ACT",
        },
        last_n_observations=12
    )

# Hors du bloc : les connexions sont fermées automatiquement
print(f"\nNombre de lignes récupérées: {len(df)}")
df.head()

In [ ]:
# Enregistrement manuel d'une structure pré-chargée
# Evite les appels d'API répétés lors d'un traitement en lot
client_batch = EurostatClient(auto_fetch_structure=False)

# Chargement explicite de la structure une seule fois
structure = client.get_dataflow_structure(dataflow="namq_10_gdp")
client_batch.register_structure(structure)

# Les appels suivants utilisent le cache sans appel réseau
df_cached = client_batch.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR", "FREQ": "Q", "NA_ITEM": "B1GQ", "UNIT": "CP_MEUR"},
    last_n_observations=4,
)
print(f"Données récupérées via cache de structure: {len(df_cached)} lignes")

### 1.11 - Gestion des erreurs <a id="section-1.11"></a>

Démonstration de la gestion des erreurs avec des requêtes invalides.

In [ ]:
# Test 1 : Dataflow inexistant
print("Test 1: Dataflow inexistant")
try:
    df = client.get_data(
        dataflow="INVALID_DATAFLOW_XYZ",
        dimensions={"GEO": "FR"},
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

# Test 2 : Dimension invalide (validation via structure)
print("\nTest 2: Nom de dimension invalide")
try:
    df = client.get_data(
        dataflow="namq_10_gdp",
        dimensions={"PAYS": ["FR"]},  # Devrait être GEO
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

# Test 3 : Dépassement de max_split_combinations
print("\nTest 3: Dépassement de max_split_combinations")
try:
    df = client.get_data(
        dataflow="namq_10_gdp",
        dimensions={
            "GEO": ["FR", "DE", "IT", "ES", "NL", "BE", "PT", "AT", "PL", "SE"],
            "NA_ITEM": ["B1GQ", "P3", "P5G", "P6", "P7", "B11"],
        },
        split_dimensions=["GEO", "NA_ITEM"],
        max_split_combinations=50,  # 10 × 6 = 60 > 50
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

print("\n✓ Tests de gestion d'erreurs terminés")

### 1.12 - Memento et conseils de performance <a id="section-1.12"></a>

Meilleures pratiques pour optimiser les téléchargements de données Eurostat.

#### 1. Rate Limiting
- Le client charge automatiquement la configuration depuis `parameters/eurostat.json` si le fichier existe
- En l'absence de configuration, aucun rate limiting n'est appliqué
- Configuration recommandée : **30 requêtes par minute**

```python
from macroforecast.datasets.core.rate_limiter import RateLimiter

# Instanciation manuelle avec rate limiting explicite
client = EurostatClient(
    rate_limiter=RateLimiter(requests=30, unit="minutes", count=1)
)
```

#### 2. Context Manager
- Toujours utiliser `with EurostatClient() as client:` en production
- Garantit la fermeture propre des connexions réseau

```python
with EurostatClient() as client:
    df = client.get_data(dataflow="namq_10_gdp", dimensions={"GEO": "FR"})
```

#### 3. Split Dimensions
- Utiliser `split_dimensions` pour les requêtes avec de nombreuses valeurs par dimension
- Génère plusieurs petites requêtes plutôt qu'une seule requête volumineuse
- Exemple : 10 pays × 6 indicateurs = 60 requêtes avec `split_dimensions=["GEO", "NA_ITEM"]`

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": ["FR", "DE", "IT", "ES", "NL", "BE"]},
    split_dimensions=["GEO"]
)
```

#### 4. Cache de structure
- Utiliser `auto_fetch_structure=False` + `register_structure()` pour les traitements en lot
- Evite un appel réseau par requête pour récupérer la structure du dataflow

```python
client = EurostatClient(auto_fetch_structure=False)
structure = client.get_structure("namq_10_gdp")
client.register_structure(structure)

# Les appels suivants n'effectuent plus d'appel réseau pour la structure
for country in countries:
    df = client.get_data(dataflow="namq_10_gdp", dimensions={"GEO": country})
```

#### 5. Datasets Comext
- Les datasets `DS-*` utilisent automatiquement l'API Comext
- Aucune configuration supplémentaire n'est requise

```python
df = client.get_data(
    dataflow="DS-045409",  # Bascule automatiquement vers l'API Comext
    dimensions={"FLOW": "1", "REPORTER": "FR"},
    last_n_observations=4,
)
```

#### 6. Formats de réponse
- **CSV** (défaut) : format SDMX-CSV 3.0, le plus robuste pour pandas
- **JSON** : JSON-stat 2.0, utile pour les intégrations JavaScript
- **TSV** : format héritage, à éviter sauf contrainte externe
- **XML** : SDMX-ML 3.0, pour les usages avancés

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR"},
    format=EurostatResponseFormat.JSON  # Format alternatif
)
```

#### 7. Filtrage temporel
- Préférer `last_n_observations` pour ne charger que les données récentes
- Utiliser `start_period` / `end_period` pour une fenêtre fixe
- Le format de période suit la norme SDMX : `"2024"`, `"2024-Q1"`, `"2024-01"`

```python
# Seulement les 4 derniers trimestres
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR", "NA_ITEM": "B1GQ"},
    last_n_observations=4
)
```

#### 8. Gestion des doublons
- Par défaut : `on_duplicate="warn"` (affiche un warning dans les logs)
- Options : `"ignore"`, `"warn"`, `"raise"`

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR"},
    on_duplicate="raise"  # Exception immédiate en cas de doublon
)
```

## Conclusion

Ce notebook a démontré toutes les fonctionnalités principales du client Eurostat :

- récupération de la structure parsée d'un dataflow via `get_dataflow_structure()` ;
- interrogation des cinq endpoints de structure SDMX via `get_structure()` (`dataflow`, `datastructure`, `dataconstraint`, `codelist`, `conceptscheme`) et utilisation du paramètre `references` pour récupérer les artefacts liés ;
- exécution de requêtes de données avec différents filtres de dimensions et de périodes ;
- gestion des formats de réponse, des requêtes splitées, des datasets Comext, du context manager et de la gestion des erreurs.

Pour plus d'informations, consulter :
- Le fichier `macroforecast/datasets/sources/eurostat.py`